# PyTO: run and inspect one topology optimization problem
Pick a **problem** and a **method** in the first code cell, then run all cells. The problem is built and optimized
exactly as the benchmark runner (`topopt_run_benchmarks.py`) does, using the same `build_benchmark` function. So a
result here is the same computation as the matching row of a full sweep.

**Methods and where their gradient comes from** (the method decides it; there is no separate autograd switch):

| Method | Gradient | Runs |
|---|---|---|
| `DENSITYMMA` | `torch.autograd` through the FE solve (adjoint) | every objective/constraint, incl. stress, mass, user-defined (`GFUNCTION`), thermo-structural |
| `DENSITYOCM` | `torch.autograd` | compliance + volume fraction only; not reliable for thermally loaded or self-weight problems |
| `PARETO` | hand-derived topological sensitivity | compliance + volume fraction (structural, thermal, body force) |
| `LEVELSET` | hand-derived shape derivative | compliance + volume fraction (structural, thermal) |

Problem categories: `StructuralTOExamples`, `ThermalTOExamples`, `ThermoStructuralTOExamples` (coupled thermal ->
thermal expansion load -> structure; see `Results/Publications/PyTO_comparison` for the paper comparisons).

Sections: 1 choose, 2 build and check the setup, 3 optimize, 4 results, 5 plots, 6 user-defined
objective, 7 full sweeps and reports.

## 1. Choose the problem and method

In [ ]:
import os, sys, time, tempfile
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pyvista; pyvista.OFF_SCREEN = True
import torch

from pyto.topopt.common import TO_METHODS, TO_QOI, FEA_MODE
from pyto.examples_benchmarks.topopt_run_benchmarks import *   # build_benchmark, problem lists, all TO example enums
from pyto.topopt.drivers.mma import topopt_mma
from pyto.topopt.drivers.oc import topopt_optimality_criteria
from pyto.topopt.drivers.pareto import topopt_pareto
from pyto.topopt.drivers.levelset import topopt_levelset

# ---------------- choose here ----------------
PROBLEM = StructuralTOExamples.MBBBeam
# PROBLEM = StructuralTOExamples.LBracketTopLoad_Stress_Vol       # stress objective (MMA only)
# PROBLEM = ThermalTOExamples.HeatPlate
# PROBLEM = ThermoStructuralTOExamples.MBBBeamNoHeat              # coupled problems: MMA recommended
METHOD = TO_METHODS.DENSITYMMA          # DENSITYMMA, DENSITYOCM, PARETO or LEVELSET
MAX_ITERS = None                        # None -> the problem's to_params.MaxIterations (what the sweep uses)
BINARIZE = True                         # the sweep binarizes MMA/OC results (not with Heaviside projection)
# ---------------------------------------------

OUT = tempfile.mkdtemp(prefix="pyto_nb_")

def show(images, titles, figsize=None):
    """Show saved PNGs side by side (the codebase plotters write PNGs)."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize or (6 * len(images), 5))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        if im is not None and os.path.exists(im): ax.imshow(Image.open(im))
        else: ax.text(0.5, 0.5, "not available", ha="center", va="center")
        ax.set_title(t, fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()

print(f"{PROBLEM.name} ({type(PROBLEM).__name__}) with {METHOD.name}")

## 2. Build the problem (same code as the benchmark runner) and check the setup

In [ ]:
fe, to_params, feaMode = build_benchmark(PROBLEM)
iters = MAX_ITERS or to_params.MaxIterations
camera = "xy" if to_params.ExtrudeZ else "iso"        # the runner's camera rule
per_node = 1 if feaMode == FEA_MODE.THERMAL else 3
print(f"  solver           : {fe.solver.name}, ndof {fe.bc.num_dofs}")
print(f"  objective        : {getattr(to_params.Objective[0], 'name', to_params.Objective[0])}")
print(f"  constraints      : {[(getattr(c[0], 'name', c[0]), c[2]) for c in to_params.Constraints]}")
print(f"  symmetry/extrude : X={to_params.XSymmetry} Y={to_params.YSymmetry} Z={to_params.ZSymmetry}, ExtrudeZ={to_params.ExtrudeZ}")
print(f"  supports / loads : {len(np.unique(np.asarray(fe.bc.fixed_dofs) // per_node))} fixed nodes, "
      f"{len(np.unique(np.nonzero(fe.bc.force)[0] // per_node))} loaded nodes, total load {fe.bc.force.sum():.4g}")
if feaMode == FEA_MODE.THERMO_STRUCTURAL:
    print(f"  thermal          : {len(fe.thermal_fea.bc.fixed_dofs)} nodes with prescribed temperature "
          f"{sorted(set(np.round(fe.thermal_fea.bc.dirichlet_values, 2)))}, reference {to_params.ThermalReferenceTemperature} C")
print(f"  max iterations   : {iters}")

setup_png = os.path.join(OUT, "problem.png")
fe.plot_mesh(title="This run: setup", plot_bc=True, camera_position=camera, save_path=setup_png)
show([setup_png], ["Setup: supports and loads"], figsize=(7, 5))

## 3. Optimize (same calls and arguments as the benchmark runner)

In [ ]:
t0 = time.time()
if METHOD == TO_METHODS.DENSITYMMA:
    u, history, success, err, nFEAs = topopt_mma(fe, to_params=to_params, maxMMAIterations=iters,
        binarize_topology=BINARIZE and not to_params.HeavisideProjection, print_progress=False)
elif METHOD == TO_METHODS.DENSITYOCM:
    u, history, success, err, nFEAs = topopt_optimality_criteria(fe, to_params=to_params, maxIterations=iters,
        binarize_topology=BINARIZE, print_progress=False)
elif METHOD == TO_METHODS.PARETO:
    u, history, success, err, nFEAs = topopt_pareto(feaMode, fe, to_params=to_params, print_progress=False)
elif METHOD == TO_METHODS.LEVELSET:
    u, history, success, err, nFEAs = topopt_levelset(feaMode, fe, to_params=to_params, maxIterations=iters)
elapsed = time.time() - t0
J = [float(np.ravel(v)[0]) for v in history["objective"]]
vf = [float(np.ravel(v)[0]) for v in history["volfrac"]]
print(f"success={success}  message={err}  FEAs={nFEAs}  time={elapsed:.1f} s")

## 4. Results
For MMA/OC with binarization the last history entry is the final 0/1 design; the one before it is the last continuous
iterate.

In [ ]:
print(f"{'run':14s} {'objective':>14s} {'volfrac':>9s} {'#FEAs':>6s}  success")
if METHOD in (TO_METHODS.DENSITYMMA, TO_METHODS.DENSITYOCM) and BINARIZE and len(J) > 1:
    print(f"{'last iterate':14s} {J[-2]:14.6g} {vf[-2]:9.4f} {'':>6s}  (continuous design, before binarization)")
print(f"{'this run':14s} {J[-1]:14.6g} {vf[-1]:9.4f} {nFEAs:6d}  {success}")


## 5. Plots: topology, fields and convergence (codebase plotters)

In [ ]:
topo_png = os.path.join(OUT, "topology.png")
fe.plot_mesh(save_path=topo_png, plot_bc=None, title=None, camera_position=camera)   # the runner's result image
show([topo_png], [f"Topology: J={J[-1]:.4g}, vf={vf[-1]:.3f}"], figsize=(7, 5))

In [ ]:
dens_png = os.path.join(OUT, "density.png"); fe.plot_pseudo_density(save_path=dens_png, title="Pseudo density")
imgs, names = [dens_png], ["Pseudo density"]
if feaMode == FEA_MODE.THERMAL:
    T_png = os.path.join(OUT, "temperature.png"); fe.plot_temperature(save_path=T_png)
    imgs += [T_png]; names += ["Temperature"]
else:
    if feaMode == FEA_MODE.THERMO_STRUCTURAL:
        T_png = os.path.join(OUT, "temperature.png"); fe.thermal_fea.plot_temperature(save_path=T_png)
        imgs += [T_png]; names += ["Temperature"]
    fe.postprocess()
    def_png = os.path.join(OUT, "deformation.png"); fe.plot_deformation(save_path=def_png)
    vm_png = os.path.join(OUT, "vonmises.png"); fe.plot_vonMisesStress(save_path=vm_png)
    imgs += [def_png, vm_png]; names += ["Deformation", "von Mises stress"]
show(imgs, names)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.set_xlabel("Iteration"); ax1.set_ylabel("objective", color="tab:blue")
ax1.plot(J, color="tab:blue"); ax1.tick_params(axis="y", labelcolor="tab:blue")
ax2 = ax1.twinx(); ax2.set_ylabel("volume fraction", color="tab:orange")
ax2.plot(vf, color="tab:orange", linestyle=":"); ax2.tick_params(axis="y", labelcolor="tab:orange")
plt.title(f"{METHOD.name}: {PROBLEM.name}"); ax1.grid(True); plt.show()

## 6. User-defined objective or constraint (`TO_QOI.GFUNCTION`, MMA only)
Write a function of the FE solution `sol`, the physical density `x` and the FE solver, using **torch operations
only**, and put it in the objective or constraint slot. Autograd differentiates it through the FE solve, so no
sensitivity has to be derived:

```python
to_params.Objective   = (TO_QOI.GFUNCTION, my_objective)
to_params.Constraints = [(TO_QOI.GFUNCTION, my_g, limit)]   # my_g <= limit (limit > 0)
                        [(TO_QOI.GFUNCTION, my_g, None)]    # my_g <= 0
```
NumPy inside the function breaks the gradient path and raises an error saying so. Example below: minimize the mean
tip deflection of the cantilever. With one point load this equals compliance / load, so the design matches the
compliance design (a quick way to check a user function).

In [ ]:
fe_u, params_u, _ = build_benchmark(StructuralTOExamples.CantileverTipLoad)
load_dofs = torch.as_tensor(np.nonzero(fe_u.bc.force)[0])
direction = torch.sign(torch.as_tensor(fe_u.bc.force[load_dofs.numpy()]))

def mean_tip_deflection(sol, x, fe_solver):
    """Mean displacement along the load direction at the loaded nodes (torch operations only)."""
    return (sol[load_dofs] * direction).mean()

params_u.Objective = (TO_QOI.GFUNCTION, mean_tip_deflection)            # volume fraction constraint unchanged
_, h_u, ok_u, msg_u, n_u = topopt_mma(fe_u, to_params=params_u, maxMMAIterations=params_u.MaxIterations,
                                      print_progress=False)
deflection = float(np.ravel(h_u["objective"][-1])[0])
print(f"mean tip deflection {deflection:.4e} m, vf {float(np.ravel(h_u['volfrac'][-1])[0]):.3f}, {n_u} FEAs, {msg_u}")
print(f"load x deflection = {np.abs(fe_u.bc.force).sum() * deflection:.4f}  (the compliance of this design)")
user_png = os.path.join(OUT, "user_objective.png")
fe_u.plot_mesh(save_path=user_png, plot_bc=None, title=None, camera_position="xy")
show([user_png], ["User objective: mean tip deflection"], figsize=(7, 4))

## 7. Full sweeps, reports and tools (run in a terminal at the repository root)
Environment used for every command: `PYTHONPATH=src MPLBACKEND=Agg PYVISTA_OFF_SCREEN=true` and the `PyToLib` Python.

**Full benchmark sweep** (every problem x method, each in its own subprocess; output in `Results/Results_<today>/`; many hours):
```bash
nohup python -m pyto.examples_benchmarks.topopt_run_benchmarks > benchmark_run.log 2>&1 &
```